# UI/UX Pro Max Agent

The UI/UX Pro Max Agent is a design intelligence assistant that generates complete design systems and provides implementation-ready UI/UX guidance across web and mobile platforms. Given a product type, style direction, and tech stack, it produces color tokens, typography scales, spacing systems, component states, and domain-specific best practices — all tailored to your framework.


In [28]:
!pip install -q aixplain

## Add your aiXplain access key

If you haven't yet, register to receive **5 credits** and get your aiXplain **access key** from the [Integrations](https://platform.aixplain.com/account/integrations) page.


In [ ]:
AixplainAPIKey  = "<YOUR_AIXPLAIN_API_KEY>" 

import os
os.environ["TEAM_API_KEY"] = AixplainAPIKey


In [ ]:
import inspect
from aixplain import Aixplain

aix = Aixplain(AixplainAPIKey)

# What is the UI/UX Pro Max Agent?

The goal of this agent is to create an AI-powered design assistant that helps developers and designers build polished, accessible interfaces faster. Given a product type and style direction, the agent generates a full design system spec and provides framework-specific implementation guidance.

## Agent Architecture

The agent uses two custom script tools:

- **UI UX Design Token Generator** — produces color tokens, typography scale, spacing rhythm, component states, and animation tokens based on the product type, visual style, and target platform.
- **UI UX Pattern Search** — returns domain-specific best practices (navigation, forms, accessibility, charts, animation) with stack-specific implementation tips.

## Agent Workflow

1. **Triage** — extract product type, stack, platform, and style direction from the request.
2. **Generate Design System** — call the token generator tool to produce a complete design spec.
3. **Search Patterns** — call the pattern search tool for domain-specific guidance.
4. **Deliver** — return design tokens, code snippets, accessibility notes, and anti-patterns to avoid.


In [ ]:
def generate_design_tokens(
    product_type: str,
    style_keywords: str,
    stack: str = "react",
    platform: str = "web",
    dark_mode: bool = False
) -> str:
    """
    Generates a structured design system specification for a UI/UX project.

    Parameters:
        product_type (str): Type of product, e.g. 'saas dashboard', 'mobile app', 'e-commerce',
                            'landing page', 'admin panel', 'social app', 'fintech app'.
        style_keywords (str): Visual style descriptors, e.g. 'minimal clean', 'bold vibrant',
                              'corporate professional', 'dark elegant', 'playful friendly'.
        stack (str): Frontend stack — one of: react, nextjs, vue, nuxt, svelte, astro,
                     flutter, react-native, swiftui, jetpack-compose, html-tailwind, shadcn.
                     Defaults to 'react'.
        platform (str): Target platform — 'web', 'ios', 'android', or 'desktop'. Defaults to 'web'.
        dark_mode (bool): Whether to include dark mode token variants. Defaults to False.

    Returns:
        str: A markdown-formatted design system specification including color tokens,
             typography scale, spacing system, component states, and accessibility notes.
    """
    import json

    # ── Palette library (subset of the 161 palettes from the original skill data)
    PALETTES = {
        "minimal clean": {
            "primary": "#2563EB", "primary-foreground": "#FFFFFF",
            "secondary": "#64748B", "secondary-foreground": "#FFFFFF",
            "accent": "#0EA5E9", "accent-foreground": "#FFFFFF",
            "background": "#FFFFFF", "surface": "#F8FAFC",
            "border": "#E2E8F0", "muted": "#94A3B8",
            "success": "#16A34A", "warning": "#D97706", "error": "#DC2626",
            "text-primary": "#0F172A", "text-secondary": "#475569",
        },
        "bold vibrant": {
            "primary": "#7C3AED", "primary-foreground": "#FFFFFF",
            "secondary": "#EC4899", "secondary-foreground": "#FFFFFF",
            "accent": "#F59E0B", "accent-foreground": "#1C1917",
            "background": "#FAFAFA", "surface": "#F4F4F5",
            "border": "#D4D4D8", "muted": "#71717A",
            "success": "#059669", "warning": "#D97706", "error": "#E11D48",
            "text-primary": "#18181B", "text-secondary": "#52525B",
        },
        "corporate professional": {
            "primary": "#1E40AF", "primary-foreground": "#FFFFFF",
            "secondary": "#374151", "secondary-foreground": "#FFFFFF",
            "accent": "#0369A1", "accent-foreground": "#FFFFFF",
            "background": "#FFFFFF", "surface": "#F9FAFB",
            "border": "#D1D5DB", "muted": "#9CA3AF",
            "success": "#15803D", "warning": "#B45309", "error": "#B91C1C",
            "text-primary": "#111827", "text-secondary": "#4B5563",
        },
        "dark elegant": {
            "primary": "#A78BFA", "primary-foreground": "#1C1C2E",
            "secondary": "#34D399", "secondary-foreground": "#1C1C2E",
            "accent": "#F472B6", "accent-foreground": "#1C1C2E",
            "background": "#0F0F1A", "surface": "#1A1A2E",
            "border": "#2D2D44", "muted": "#6B7280",
            "success": "#34D399", "warning": "#FBBF24", "error": "#F87171",
            "text-primary": "#F1F5F9", "text-secondary": "#94A3B8",
        },
        "playful friendly": {
            "primary": "#EF4444", "primary-foreground": "#FFFFFF",
            "secondary": "#F97316", "secondary-foreground": "#FFFFFF",
            "accent": "#FACC15", "accent-foreground": "#1C1917",
            "background": "#FFFBF5", "surface": "#FEF3C7",
            "border": "#FDE68A", "muted": "#92400E",
            "success": "#16A34A", "warning": "#D97706", "error": "#DC2626",
            "text-primary": "#1C1917", "text-secondary": "#78716C",
        },
    }

    # ── Typography library (57 font pairings from the original skill data)
    TYPOGRAPHY = {
        "saas dashboard": {"heading": "Inter", "body": "Inter", "mono": "JetBrains Mono"},
        "e-commerce":     {"heading": "Playfair Display", "body": "Lato", "mono": "Courier Prime"},
        "landing page":   {"heading": "Cal Sans", "body": "Inter", "mono": "Fira Code"},
        "admin panel":    {"heading": "DM Sans", "body": "DM Sans", "mono": "Roboto Mono"},
        "mobile app":     {"heading": "SF Pro Display", "body": "SF Pro Text", "mono": "SF Mono"},
        "social app":     {"heading": "Nunito", "body": "Nunito Sans", "mono": "Source Code Pro"},
        "fintech app":    {"heading": "IBM Plex Sans", "body": "IBM Plex Sans", "mono": "IBM Plex Mono"},
        "default":        {"heading": "Inter", "body": "Inter", "mono": "JetBrains Mono"},
    }

    # ── Stack-specific implementation notes
    STACK_NOTES = {
        "react":          "Use CSS custom properties with a ThemeProvider; Tailwind CSS `@layer base` for tokens.",
        "nextjs":         "Place tokens in `styles/globals.css`; use `next-themes` for dark mode switching.",
        "vue":            "Define tokens in `assets/css/tokens.css`; use `vueuse/core` `useColorMode`.",
        "nuxt":           "Configure via `nuxt.config.ts` CSS array; `@nuxtjs/color-mode` module for dark mode.",
        "svelte":         "Declare tokens in `:root` inside `app.css`; toggle dark mode via class on `<html>`.",
        "flutter":        "Define `ThemeData` with `ColorScheme.fromSeed`; use `Theme.of(context)` everywhere.",
        "react-native":   "Create a `theme.ts` constants file; use `useColorScheme` for dark/light switching.",
        "swiftui":        "Use `Color(\"tokenName\")` from Assets catalog; support `@Environment(\.colorScheme)`.",
        "html-tailwind":  "Extend `tailwind.config.js` colors; use `darkMode: 'class'` strategy.",
        "shadcn":         "Tokens live in `globals.css` as HSL variables; `cn()` utility for variant merging.",
        "default":        "Define tokens as CSS custom properties in `:root`; toggle dark mode via class.",
    }

    # ── Touch targets by platform
    TOUCH_TARGETS = {
        "ios": "44×44pt (Apple HIG minimum)",
        "android": "48×48dp (Material Design minimum)",
        "web": "44×44px recommended (WCAG 2.5.5)",
        "desktop": "32×32px minimum; hover states required",
    }

    # ── Resolve inputs
    style_key = next((k for k in PALETTES if k in style_keywords.lower()), "minimal clean")
    palette = PALETTES[style_key]
    fonts = TYPOGRAPHY.get(product_type.lower(), TYPOGRAPHY["default"])
    stack_note = STACK_NOTES.get(stack.lower(), STACK_NOTES["default"])
    touch_target = TOUCH_TARGETS.get(platform.lower(), TOUCH_TARGETS["web"])

    # ── Dark mode overrides
    dark_tokens = ""
    if dark_mode:
        dark_tokens = """
### Dark Mode Token Overrides
```
--background:       #0F172A
--surface:          #1E293B
--border:           #334155
--text-primary:     #F1F5F9
--text-secondary:   #94A3B8
--muted:            #475569
```
Apply by toggling `class="dark"` on `<html>`. All semantic tokens automatically resolve.
"""

    output = f"""# Design System — {product_type.title()} ({style_keywords.title()})
**Stack:** {stack} | **Platform:** {platform} | **Style:** {style_key}

---

## Color Tokens
```
--primary:              {palette['primary']}
--primary-foreground:   {palette['primary-foreground']}
--secondary:            {palette['secondary']}
--secondary-foreground: {palette['secondary-foreground']}
--accent:               {palette['accent']}
--accent-foreground:    {palette['accent-foreground']}
--background:           {palette['background']}
--surface:              {palette['surface']}
--border:               {palette['border']}
--muted:                {palette['muted']}
--success:              {palette['success']}
--warning:              {palette['warning']}
--error:                {palette['error']}
--text-primary:         {palette['text-primary']}
--text-secondary:       {palette['text-secondary']}
```
{dark_tokens}
## Typography Scale
**Heading font:** {fonts['heading']}  
**Body font:** {fonts['body']}  
**Mono font:** {fonts['mono']}

```
display:   3rem   / 1.1  / -0.02em  (page heroes)
h1:        2.25rem / 1.2  / -0.01em
h2:        1.875rem / 1.25 / -0.01em
h3:        1.5rem  / 1.3  / 0
h4:        1.25rem / 1.4  / 0
body-lg:   1.125rem / 1.6  / 0       (comfortable reading)
body:      1rem    / 1.5  / 0        (base — never go below)
body-sm:   0.875rem / 1.5  / 0
caption:   0.75rem / 1.4  / 0.01em
mono:      0.875rem / 1.6  / 0       (code, data)
```

## Spacing System (4dp/8dp rhythm)
```
space-1:   4px   (tight gaps — icon padding)
space-2:   8px   (component internal padding)
space-3:   12px
space-4:   16px  (standard unit)
space-5:   20px
space-6:   24px  (section padding)
space-8:   32px
space-10:  40px
space-12:  48px  (section separation)
space-16:  64px
space-20:  80px  (hero / major sections)
space-24:  96px
```

## Border Radius
```
radius-sm:  4px   (inputs, badges)
radius-md:  8px   (cards, buttons)
radius-lg:  12px  (modals, panels)
radius-xl:  16px  (sheets, drawers)
radius-full: 9999px (pills, avatars)
```

## Shadow Tokens
```
shadow-sm:  0 1px 2px rgba(0,0,0,0.05)
shadow-md:  0 4px 6px rgba(0,0,0,0.07), 0 2px 4px rgba(0,0,0,0.05)
shadow-lg:  0 10px 15px rgba(0,0,0,0.10), 0 4px 6px rgba(0,0,0,0.05)
shadow-xl:  0 20px 25px rgba(0,0,0,0.10), 0 10px 10px rgba(0,0,0,0.04)
```

## Component States
```
default:   base token values
hover:     lighten/darken primary by 10%; scale(1.01) for cards
active:    darken primary by 15%; scale(0.98)
focus:     2px outline offset 2px — color: primary (never remove outline)
disabled:  opacity: 0.4; cursor: not-allowed; pointer-events: none
loading:   skeleton shimmer or spinner; preserve layout dimensions
error:     border-color: error; show message below field (not tooltip)
success:   border-color: success; inline confirmation message
empty:     centered illustration + CTA; never a blank container
```

## Animation Tokens
```
duration-fast:    150ms  (micro interactions, hover)
duration-normal:  250ms  (state transitions)
duration-slow:    350ms  (modals, drawers)
easing-default:   cubic-bezier(0.4, 0, 0.2, 1)
easing-spring:    cubic-bezier(0.34, 1.56, 0.64, 1)
easing-exit:      cubic-bezier(0.4, 0, 1, 1)
```
Use `transform` only (no `top`/`left`/`width` animations). Respect `prefers-reduced-motion`.

## Touch & Interaction
- Minimum touch target: **{touch_target}**
- Minimum spacing between targets: **8px**
- Interaction feedback: within **80–150ms**
- All interactive elements: focus ring + keyboard navigable

## Accessibility Checklist
- [ ] Text contrast ≥ 4.5:1 (body), ≥ 3:1 (large text / UI components)
- [ ] All images have meaningful `alt` text; decorative images use `alt=""`
- [ ] Focus order matches visual reading order
- [ ] No information conveyed by color alone
- [ ] All form inputs have visible `<label>` elements
- [ ] ARIA roles only where semantic HTML is insufficient

## Stack Implementation
**{stack}:** {stack_note}

## Anti-Patterns to Avoid
- Emoji as structural icons — use Lucide, Heroicons, or platform-native icon sets
- Hardcoded hex values in components — always reference semantic tokens
- `px` for font sizes — use `rem` for user accessibility scaling
- Removing focus outlines without providing a replacement
- `margin: auto` on flex children without understanding stacking context
- Fixed pixel heights on text containers — use `min-height` instead
"""
    return output.strip()

In [ ]:
def search_design_patterns(query: str, domain: str = "ux", stack: str = "") -> str:
    """
    Returns design guidance and patterns for a specific domain or technology stack.

    Parameters:
        query (str): What you want to design or improve, e.g. 'navigation bar mobile',
                     'data table sorting', 'onboarding flow', 'checkout form'.
        domain (str): Design domain to search — one of:
                      'ux' (UX guidelines), 'ui' (UI patterns), 'accessibility',
                      'navigation', 'forms', 'charts', 'typography', 'color', 'animation'.
                      Defaults to 'ux'.
        stack (str): Optional stack filter to get framework-specific implementation tips.
                     One of: react, nextjs, vue, flutter, react-native, swiftui, tailwind, shadcn.

    Returns:
        str: Relevant design patterns, best practices, and implementation guidance.
    """

    PATTERNS = {
        "navigation": """
## Navigation Patterns

### Mobile (iOS / Android / React Native)
- Bottom tab bar: max 5 items; always show labels; active state must be unmistakably distinct
- Use a hamburger menu only for secondary nav that doesn't fit in tabs
- Swipe-back gesture: never block it; use it for secondary navigation flow
- Deep linking: every screen reachable via URL / universal link
- Back button: always predictable — goes to the previous screen, not home

### Web / Desktop
- Top nav: left-aligned logo, center or right-aligned links, sticky on scroll
- Sidebar: collapsible; persist collapsed state to localStorage
- Breadcrumbs: show on pages 3+ levels deep; structured data markup
- Active state: use both color AND a shape indicator (underline, pill, border)
- Skip-to-content link: first focusable element for keyboard users

### State Patterns
- Loading: skeleton that mirrors final nav layout — never a spinner in the nav bar
- Notification badge: max 2-digit count; use '+' for 99+; red dot for urgency only
- Overflow: show a '...' more menu; never truncate nav items without disclosure
""",
        "forms": """
## Form Design Patterns

### Layout
- Single column: always preferred for mobile and short forms
- Multi-column: only for logically related pairs (first name / last name)
- Group related fields with a visible separator or heading
- Place primary CTA button at the bottom right; secondary action to its left

### Labels & Inputs
- Always use a visible `<label>` — never placeholder-only labels
- Floating labels: acceptable if there's always a visible placeholder hint below
- Required indicator: asterisk (*) with legend "* required" at the top
- Input height: minimum 44px; padding 12px horizontal, 10px vertical
- Character count: show below input when a hard limit exists

### Validation
- Validate on blur (after user leaves field), not on every keystroke
- Error message: below the field in error color; include an icon; be specific
  - Bad:  "Invalid input"
  - Good: "Email must include @ and a domain, e.g. user@example.com"
- Inline success: green checkmark on blur when field is valid
- Submit error: summary at the top of the form + scroll into view

### Accessibility
- `aria-required="true"` on required fields
- `aria-describedby` linking input to its error message element
- Autofocus the first field on modal forms
- Trap focus inside modals; return focus to trigger on close
""",
        "ux": """
## UX Guidelines (Top 20 of 99)

1. **Progressive disclosure** — show only what's needed now; reveal complexity on demand
2. **Fitts's Law** — make frequent actions large and close to the current pointer position
3. **Hick's Law** — reduce decision points; fewer options = faster decisions
4. **Recognition over recall** — show options; don't make users memorize
5. **Error prevention** — disable actions that would cause errors before they're triggered
6. **Undo over confirm** — prefer reversible actions over destructive confirmation dialogs
7. **Feedback within 100ms** — any tap/click must have a visual response immediately
8. **Skeleton screens** — always preferred over spinners for content-heavy layouts
9. **Empty states** — every list/grid must have an empty state with a clear CTA
10. **Onboarding** — show value before asking for commitment (account creation)
11. **Gestalt proximity** — group related elements; use whitespace to separate unrelated ones
12. **Visual hierarchy** — one primary action per screen; secondary and tertiary actions de-emphasized
13. **Consistent affordances** — same interaction = same visual treatment every time
14. **Contextual help** — tooltips and hints near the relevant field, not in a separate FAQ
15. **Scroll depth** — critical content and CTAs above the fold on mobile
16. **Gesture shortcuts** — swipe-to-delete, long-press for context menu; always provide a visible alternative
17. **Confirmation patterns** — only confirm irreversible, high-stakes actions (delete, payment)
18. **Search UX** — show recent searches, auto-suggest, handle zero results gracefully
19. **Loading states** — indicate progress; give an estimated time for operations > 3 seconds
20. **Accessibility first** — design for keyboard, screen reader, and reduced-motion from the start
""",
        "accessibility": """
## Accessibility Patterns (WCAG AA)

### Contrast
- Normal text (< 18pt / < 14pt bold): **4.5:1** minimum
- Large text (≥ 18pt / ≥ 14pt bold): **3:1** minimum
- UI components and icons: **3:1** minimum
- Test in both light and dark mode independently

### Keyboard
- All interactive elements reachable via Tab
- Logical focus order matching visual reading order
- Visible focus indicator (2px outline minimum)
- Escape key closes any modal, tooltip, or dropdown
- Arrow keys for compound widgets (menus, tabs, sliders)

### Screen Readers
- Semantic HTML first: `<button>`, `<nav>`, `<main>`, `<article>`
- Meaningful link text: not "click here" or "read more"
- Images: `alt` text for informative images; `alt=""` for decorative
- Live regions: `aria-live="polite"` for dynamic content updates
- Form errors: `aria-describedby` linking input to error message

### Motion
- `@media (prefers-reduced-motion: reduce)` — disable or reduce all animations
- No content that flashes more than 3 times per second
""",
        "charts": """
## Chart & Data Visualization Patterns

### Choosing the Right Chart
| Goal | Chart Type |
|---|---|
| Compare categories | Bar / Column chart |
| Show trend over time | Line chart |
| Show proportions | Pie / Donut (max 5 slices) |
| Show distribution | Histogram / Box plot |
| Show correlation | Scatter plot |
| Show progress to goal | Radial / Linear gauge |
| Show geographic data | Choropleth map |
| Compare many metrics | Radar / Spider chart |

### Design Principles
- Use accessible color pairs — never rely on color alone to distinguish series
- Always include a legend when multiple series exist
- Tooltips: show exact value on hover/tap; position above the chart element
- Y-axis: start at 0 for bar charts; annotate if truncated on line charts
- Provide a table alternative for screen readers (`<table>` with same data)
- Responsive: charts must reflow at mobile widths; hide minor grid lines on small screens

### Recommended Libraries
- React: Recharts, Victory, Nivo, Chart.js via react-chartjs-2
- Vue: Vue-Chartjs, ApexCharts Vue
- Flutter: fl_chart, syncfusion_flutter_charts
- Vanilla: D3.js, Apache ECharts
""",
        "animation": """
## Animation & Motion Patterns

### Timing Rules
- Micro-interactions (hover, tap feedback): 100–150ms
- State transitions (toggle, expand): 200–250ms
- Page/screen transitions: 300–400ms
- Never exceed 500ms for UI animations

### Easing
- Entering elements: ease-out (decelerate into resting position)
- Exiting elements: ease-in (accelerate out of view)
- Bouncy/spring: use sparingly for playful products only

### What to Animate (and What Not To)
- ✅ `transform: translate/scale/rotate` — GPU composited, smooth
- ✅ `opacity` — GPU composited
- ❌ `width`, `height`, `top`, `left`, `margin` — triggers layout recalc
- ❌ Animating more than 3 things simultaneously

### Reduced Motion
```css
@media (prefers-reduced-motion: reduce) {
  *, *::before, *::after {
    animation-duration: 0.01ms !important;
    transition-duration: 0.01ms !important;
  }
}
```
""",
    }

    STACK_PATTERNS = {
        "react": """
## React Implementation Patterns
- Use `clsx` or `cn()` (shadcn) for conditional class merging
- Component variants: `cva()` (class-variance-authority) for type-safe variant props
- Lazy load heavy components: `React.lazy` + `<Suspense>` with skeleton fallback
- Avoid inline styles; use CSS modules or Tailwind utilities
- `useId()` for generating accessible label-input associations
- Memoize expensive renders: `React.memo`, `useMemo`, `useCallback` — but profile first
""",
        "react-native": """
## React Native Implementation Patterns
- Use `StyleSheet.create()` for all styles — avoids object recreation on re-render
- `Pressable` over `TouchableOpacity` — more flexible feedback customization
- Always wrap screens in `<SafeAreaView>` from `react-native-safe-area-context`
- `KeyboardAvoidingView` on all screens with form inputs
- Image: use `<FastImage>` for performance; always set `width` and `height`
- Navigation: React Navigation v6 — use typed routes to prevent runtime crashes
- Platform-specific styles: `Platform.select({ ios: {...}, android: {...} })`
""",
        "flutter": """
## Flutter Implementation Patterns
- Use `Theme.of(context).colorScheme` everywhere — never hardcode colors
- `MediaQuery.of(context).padding` for safe area insets
- `LayoutBuilder` for responsive breakpoints within widgets
- `AnimatedContainer`, `AnimatedOpacity` for declarative animations
- `Hero` widget for shared element transitions between routes
- Semantics widget for accessibility labels on custom widgets
- Const constructors wherever possible for performance
""",
        "swiftui": """
## SwiftUI Implementation Patterns
- Use semantic colors: `.primary`, `.secondary`, `Color("tokenName")` from asset catalog
- `@Environment(\.colorScheme)` for dark/light mode detection
- `accessibilityLabel()` and `accessibilityHint()` on all interactive views
- `.dynamicTypeSize(.xSmall ... .accessibility3)` for text scaling support
- `GeometryReader` for responsive sizing — use sparingly, prefer flexible frames
- `.safeAreaInset` for safe area content placement
- `withAnimation(.easeInOut(duration: 0.25))` for state-driven transitions
""",
        "tailwind": """
## Tailwind CSS Patterns
- Extend `tailwind.config.js` with semantic color tokens — avoid arbitrary values in JSX
- Use `@layer components` for reusable component classes
- `darkMode: 'class'` strategy + toggle `dark` class on `<html>` for user control
- `clamp()` for fluid typography: `text-[clamp(1rem,2.5vw,1.5rem)]`
- Container queries with `@tailwindcss/container-queries` plugin
- Never use `!important` utilities unless overriding third-party styles
""",
    }

    domain_key = domain.lower()
    result = PATTERNS.get(domain_key, PATTERNS["ux"])

    if stack and stack.lower() in STACK_PATTERNS:
        result += "\n---\n" + STACK_PATTERNS[stack.lower()]

    # Simple keyword boost — surface the most relevant section if query matches a domain
    query_lower = query.lower()
    for key in PATTERNS:
        if key in query_lower and key != domain_key:
            result += f"\n---\n**Also relevant — {key.title()} patterns:**\n" + PATTERNS[key]
            break

    return result.strip()

In [ ]:
SCRIPT_INTEGRATION_ID = "688779d8bfb8e46c273982ca"

design_tokens_tool = aix.Tool(
    name="UI UX Design Token Generator",
    integration=SCRIPT_INTEGRATION_ID,
    config={"code": inspect.getsource(generate_design_tokens), "function_name": "generate_design_tokens"},
)
design_tokens_tool.save()

design_patterns_tool = aix.Tool(
    name="UI UX Pattern Search",
    integration=SCRIPT_INTEGRATION_ID,
    config={"code": inspect.getsource(search_design_patterns), "function_name": "search_design_patterns"},
)
design_patterns_tool.save()


# Build Your Agent

To create an agent, define:

* A unique **name** and **description** for its purpose.
* The **tools** it will use — here, two custom script tools for generating design tokens and searching patterns.
* The **instructions** that guide the agent's behaviour and decision-making.

In [ ]:
uiux_agent = aix.Agent(
    name="UI/UX Pro Max Agent",
    description="Design intelligence agent that generates design systems, reviews UI/UX, and provides implementation-ready guidance across web and mobile stacks.",
    instructions="""
    You are an expert UI/UX design intelligence system covering 50+ styles, 161 color palettes,
    57 font pairings, 99 UX guidelines, and 25 chart types across 10 technology stacks.

    ## Your Workflow

    ### Step 1 — Triage
    When given a design request, extract:
    - Product type (saas dashboard, e-commerce, landing page, mobile app, admin panel, fintech, social app)
    - Technology stack (react, nextjs, vue, nuxt, svelte, flutter, react-native, swiftui, tailwind, shadcn)
    - Target platform (web, ios, android, desktop)
    - Style direction (minimal clean, bold vibrant, corporate professional, dark elegant, playful friendly)
    - Accessibility level (always target WCAG AA minimum)
    If any are missing, make a reasonable inference and state your assumption.

    ### Step 2 — Generate Design System
    Use the "UI UX Design Token Generator" tool to produce a full design system spec.
    Always do this before giving any design recommendations.

    ### Step 3 — Search Patterns
    Use the "UI UX Pattern Search" tool for domain-specific guidance:
    - For navigation questions: domain='navigation'
    - For form design: domain='forms'
    - For charts/data viz: domain='charts'
    - For motion/animation: domain='animation'
    - For accessibility review: domain='accessibility'
    - General UX questions: domain='ux'
    Always pass the stack parameter to get framework-specific implementation tips.

    ### Step 4 — Deliver
    Provide:
    1. The design system tokens (from Step 2)
    2. Specific component or pattern recommendations (from Step 3)
    3. Implementation code snippet in the user's stack
    4. Accessibility notes
    5. Anti-patterns to avoid for this specific use case

    ## 10 Priority Rules (Always Apply)

    1. ACCESSIBILITY (CRITICAL) — 4.5:1 contrast, keyboard navigation, alt text, screen reader support.
       Never ship without these.

    2. TOUCH & INTERACTION (CRITICAL) — 44×44pt minimum touch targets, 8px+ spacing between them,
       visual feedback within 80–150ms.

    3. PERFORMANCE (HIGH) — WebP/AVIF images, lazy loading, prevent layout shift, <100ms interactions.

    4. STYLE SELECTION (HIGH) — Match visual style to product type. Use SVG icons, never emoji.
       Consistent shadow and effect treatment.

    5. LAYOUT & RESPONSIVE (HIGH) — Mobile-first, systematic breakpoints, no horizontal scroll,
       viewport meta tag, safe-area padding.

    6. TYPOGRAPHY & COLOR (MEDIUM) — 16px base body, 1.5 line-height, semantic color tokens,
       verified contrast in light AND dark mode.

    7. ANIMATION (MEDIUM) — 150–300ms duration, transforms only, cause-effect relationships,
       respect prefers-reduced-motion.

    8. FORMS & FEEDBACK (MEDIUM) — Visible labels, errors below fields, loading indicators,
       success states, clear error recovery.

    9. NAVIGATION (HIGH) — Bottom nav max 5 items, predictable back behavior, deep linking,
       clear active state indicators.

    10. CHARTS & DATA (LOW) — Match chart type to data, accessible color pairs, legends,
        tooltips, table alternatives.

    ## Common Mistakes to Flag
    - Emoji used as structural icons
    - Hardcoded color values instead of tokens
    - Missing focus outlines
    - Placeholder-only form labels (no visible <label>)
    - Fixed pixel heights on text containers
    - Contrast failures not checked in dark mode
    - Touch targets below minimum size
    - Missing empty, loading, and error states

    ## Pre-Delivery Checklist
    Before finalizing any recommendation, confirm:
    - No emoji icons in the design
    - Consistent icon family throughout
    - Semantic color tokens used everywhere
    - Tap feedback within 80–150ms
    - Touch targets ≥ 44pt
    - Text contrast 4.5:1 in both themes
    - Safe areas respected
    - All states defined (empty, loading, error, success, disabled)
    """,
    tools=[
        design_tokens_tool,
        design_patterns_tool,
    ],
)

print(f"Agent created: {uiux_agent.id}")
uiux_agent.save()


# Run your Agent

To ensure your agent is performing as expected, run it using the `run` method with sample inputs. Analyze the outputs, verify their accuracy, and debug any issues by inspecting the agent's configurations, tools, and instructions.


In [ ]:
result = uiux_agent.run(
    """I'm building a fintech mobile app for iOS using React Native. 
    I want a professional, trustworthy feel. Generate a complete design system."""
)


In [ ]:
print(result.data.output)


In [ ]:
result = uiux_agent.run(
    "I have a React dashboard. Each row in my data table has a delete button (16x16px icon, no label). "
    "The table uses #888 text on #fff background. What UX and accessibility issues should I fix?"
)
print(result.data.output)


In [ ]:
result = uiux_agent.run(
    "How should I implement a bottom navigation bar in Flutter with 4 tabs, "
    "dark mode support and accessibility labels?"
)
print(result.data.output)


In [ ]:
result = uiux_agent.run(
    "Can you also give me chart recommendations for displaying spending analytics in the same app?",
    session_id=result.data.session_id
)
print(result.data.output)


# Saving the Agent

Once you are happy with your agent, save it to access the agent endpoints.


In [ ]:
uiux_agent.save()


aiXplain empowers you to seamlessly build, customize, and deploy intelligent agents tailored to your specific needs. Whether you're creating standalone agents or advanced multi-agent systems, the platform provides a robust toolkit for integrating cutting-edge AI capabilities into your workflows. To learn more, visit [aiXplain](https://aixplain.com).
